# 01 · O*NET occupation features

Part 1 of 5. Builds one feature vector per O*NET occupation and sanity-checks nearest neighbors.

**Reads:** `data/raw/db_31_0_csv/` (abilities, transferable_skills, knowledge, essential_skills)
**Writes:** `data/derived/job-features.parquet` (codes as index, used by 02–03), `job-features-title.parquet`, `onet-titles.parquet`; `data/interim/job-similarity-title.parquet` (large, regenerated)

**Decisions**
- Importance scale only, rows flagged `Recommend Suppress == 'Y'` dropped.
- Columns standardized before cosine similarity, because generic abilities (oral comprehension) have high means and little spread.
- Abilities alone can't separate desk jobs (software developers land next to tax preparers); adding skills and knowledge fixes it. The ranker baseline in 03 uses abilities + 25 transferable skills + knowledge (110 features).
- The survey basis chosen later (04) is different: knowledge + the 10 essential skills.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Abilities only

Physical and medical jobs separate cleanly. Desk jobs collapse together because abilities carry no domain.

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

abilities = pd.read_csv("../data/raw/db_31_0_csv/abilities.csv")
# print(abilities.head())
# print(abilities.shape)
# print(abilities.nunique())

ai = abilities.loc[abilities['Scale Name'] == 'Importance'].copy()
ai = ai.loc[abilities['Recommend Suppress'] == 'N'].copy()
ai.reset_index(drop=True, inplace=True)
aipivot = ai.pivot(index='O*NET-SOC Code', columns='Element Name', values='Data Value')

aipivot.head()
title_map = ai.set_index('O*NET-SOC Code')['Title'].to_dict()

ai_title= aipivot.copy().rename(index=title_map, inplace=True)
aipivot.head()
aipivot.describe().T.sort_values('mean')

X = StandardScaler().fit_transform(aipivot)
S = pd.DataFrame(cosine_similarity(X), index=aipivot.index, columns=aipivot.index)

def neighbors(code, k=10):
    s = S[code].drop(code).sort_values(ascending=False).head(k)
    return pd.DataFrame({'title': s.index.map(title_map), 'sim': s.round(3)})

for code in ['15-1252.00', '19-2031.00', '43-4051.00', '47-2111.00', '29-1141.00']:
    print(title_map[code])
    print(neighbors(code), '\n')




Software Developers
                                                            title    sim
O*NET-SOC Code                                                          
15-1243.01                           Data Warehousing Specialists  0.914
13-1141.00      Compensation, Benefits, and Job Analysis Speci...  0.862
43-4021.00                                  Correspondence Clerks  0.855
15-1242.00                                Database Administrators  0.847
13-2082.00                                          Tax Preparers  0.840
11-3111.00                     Compensation and Benefits Managers  0.839
15-2051.02                                 Clinical Data Managers  0.835
15-1251.00                                   Computer Programmers  0.834
19-4061.00                     Social Science Research Assistants  0.829
13-1023.00      Purchasing Agents, Except Wholesale, Retail, a...  0.829 

Chemists
                                                      title    sim
O*NET-SOC Code            

## Abilities + skills + knowledge

Knowledge adds the domain axis (Computers and Electronics, Chemistry, Medicine). Software developers now neighbor QA testers and web developers.

In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity


def load_importance(name):
    df = pd.read_csv(f"../data/raw/db_31_0_csv/{name}.csv")
    df = df[(df["Scale Name"] == "Importance") & (df["Recommend Suppress"] != "Y")]
    return df.pivot(index="O*NET-SOC Code", columns="Element Name", values="Data Value")


tables = [load_importance(n) for n in ["abilities", "transferable_skills", "knowledge"]]
feat = pd.concat(tables, axis=1, join="inner")
print(feat.shape)  # expect ~870 x 120
feat = feat.fillna(feat.mean())

X = StandardScaler().fit_transform(feat)
S = pd.DataFrame(cosine_similarity(X), index=feat.index, columns=feat.index)

for code in ["15-1252.00", "19-2031.00", "43-4051.00", "47-2111.00", "29-1141.00"]:
    print(title_map[code])
    print(neighbors(code), "\n")

# feat.to_parquet("../data/derived/job-features.parquet")
feat.rename(index=title_map).rename_axis('title').reset_index() \
    .to_parquet('../data/derived/job-features-title.parquet', index=False)

# S.to_parquet("../data/interim/job-similarity.parquet")
S.rename(index=title_map, columns=title_map).to_parquet("../data/interim/job-similarity-title.parquet")

(910, 110)
Software Developers
                                                          title    sim
O*NET-SOC Code                                                        
15-1243.01                         Data Warehousing Specialists  0.863
15-1253.00      Software Quality Assurance Analysts and Testers  0.840
15-1299.07                                 Blockchain Engineers  0.838
15-1243.00                                  Database Architects  0.836
15-1242.00                              Database Administrators  0.816
15-1254.00                                       Web Developers  0.811
15-1211.00                            Computer Systems Analysts  0.794
15-1251.00                                 Computer Programmers  0.791
15-1255.00                  Web and Digital Interface Designers  0.778
15-1299.08                Computer Systems Engineers/Architects  0.774 

Chemists
                                                            title    sim
O*NET-SOC Code                   

/tmp/ipykernel_865/2646487891.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feat.rename(index=title_map).rename_axis('title').reset_index() \


## O*NET 31.0 skills layout

31.0 split the old 35-item Skills table into 10 essential skills and 25 transferable skills.

In [4]:
for name in ['essential_skills', 'transferable_skills']:
    t = pd.read_csv(f'../data/raw/db_31_0_csv/{name}.csv')
    print(name, t['Element Name'].nunique(), 'elements; scales:', t['Scale Name'].unique())
    print(sorted(t['Element Name'].unique()), '\n')

essential_skills 10 elements; scales: <ArrowStringArray>
['Importance', 'Level']
Length: 2, dtype: str
['Active Learning', 'Active Listening', 'Critical Thinking', 'Learning Strategies', 'Mathematics', 'Monitoring', 'Reading Comprehension', 'Science', 'Speaking', 'Writing'] 



transferable_skills 25 elements; scales: <ArrowStringArray>
['Importance', 'Level']
Length: 2, dtype: str
['Complex Problem Solving', 'Coordination', 'Equipment Maintenance', 'Equipment Selection', 'Installation', 'Instructing', 'Judgment and Decision Making', 'Management of Financial Resources', 'Management of Material Resources', 'Management of Personnel Resources', 'Negotiation', 'Operation and Control', 'Operations Analysis', 'Operations Monitoring', 'Persuasion', 'Programming', 'Quality Control Analysis', 'Repairing', 'Service Orientation', 'Social Perceptiveness', 'Systems Analysis', 'Systems Evaluation', 'Technology Design', 'Time Management', 'Troubleshooting'] 



## Save hand-off files

In [5]:
out = '../data/derived/'
feat.reset_index().to_parquet(out + 'job-features.parquet', index=False)
(pd.Series(title_map, name='title').rename_axis('O*NET-SOC Code').reset_index()
   .to_parquet(out + 'onet-titles.parquet', index=False))
print(feat.shape, 'saved')

(910, 110) saved


/tmp/ipykernel_865/819909613.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feat.reset_index().to_parquet(out + 'job-features.parquet', index=False)
